In [4]:
from pathlib import Path
import sys

current_dir = Path.cwd()

if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate project root containing 'src'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: e:\github\BTC-and-ETH-Data-Analyst-and-Prediction


In [5]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from src.preprocessing import (
    clean_market_data,
    temporal_split
)

from src.feature_engineering import (
    add_technical_indicators,
)

from src.model import build_lstm_model

In [6]:
FEATURE_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "SMA_10",
    "SMA_30",
    "EMA_5",
    "EMA_20",
    "RSI_14",
    "MACD",
    "MACD_Signal"
]

TARGET_COLUMNS = [
    "Open",
    "High",
    "Low",
    "Close"
]

ASSET = "ETH-USD"

In [7]:
data_path = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "crypto_sample.csv"
)

In [ ]:
if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_path}"
    )

# 1. Read dataset
df = pd.read_csv(data_path)

print("Original dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# 2. Clean dataset
df = clean_market_data(df)

# Standardize asset names
df["Asset"] = (
    df["Asset"]
    .astype(str)
    .str.strip()
    .str.upper()
)

ASSET = ASSET.strip().upper()

# 3. Check assets before calculating indicators
available_assets = df["Asset"].unique().tolist()

print("Available assets before indicators:", available_assets)

if ASSET not in available_assets:
    raise ValueError(
        f"{ASSET} does not exist in the dataset. "
        f"Available assets: {available_assets}"
    )

# 4. Select the asset first
asset_df = (
    df.loc[df["Asset"] == ASSET]
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    f"Rows for {ASSET} before indicators:",
    len(asset_df)
)

# SMA_30 requires at least 30 rows.
# Hyperparameter analysis with validation and time_steps=30
# should preferably use about one year of daily data.
if len(asset_df) < 60:
    raise ValueError(
        f"Only {len(asset_df)} rows are available for {ASSET}. "
        "At least 60 rows are required, and approximately "
        "365 daily rows are recommended for hyperparameter analysis."
    )

# 5. Calculate technical indicators
asset_df = add_technical_indicators(asset_df)

print(
    f"Rows for {ASSET} after indicators:",
    len(asset_df)
)

if asset_df.empty:
    raise ValueError(
        f"All {ASSET} rows were removed while calculating "
        "technical indicators. Check missing values and make "
        "sure sufficient chronological data is available."
    )

print("Selected asset:", ASSET)
print("Available observations:", len(asset_df))

asset_df.head()

ValueError: No data found for ETH-USD. Available assets: []

In [ ]:
train_df, validation_df, test_df = temporal_split(
    asset_df,
    train_ratio=0.80,
    validation_ratio=0.10
)

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

In [ ]:
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

feature_scaler.fit(
    train_df[FEATURE_COLUMNS]
)

target_scaler.fit(
    train_df[TARGET_COLUMNS]
)

train_features = feature_scaler.transform(
    train_df[FEATURE_COLUMNS]
)

train_targets = target_scaler.transform(
    train_df[TARGET_COLUMNS]
)

validation_features = feature_scaler.transform(
    validation_df[FEATURE_COLUMNS]
)

validation_targets = target_scaler.transform(
    validation_df[TARGET_COLUMNS]
)

In [ ]:
BASELINE_CONFIG = {
    "time_steps": 10,
    "units": 128,
    "learning_rate": 0.0005,
    "batch_size": 32,
    "max_epochs": 100
}

max_epochs = 100
patience = 10

In [ ]:
PARAMETER_GRID = {
    "time_steps": [10, 30, 60],
    "units": [32, 64, 128, 256],
    "learning_rate": [0.001, 0.0005, 0.0001],
    "batch_size": [16, 32, 64, 128]
}

In [ ]:
train_features
train_targets
validation_features
validation_targets
import numpy as np
import tensorflow as tf

from keras.callbacks import EarlyStopping
from keras import backend as K

from src.feature_engineering import create_sequences
from src.utils import set_random_seed


def run_experiment(
    train_features,
    train_targets,
    validation_features,
    validation_targets,
    time_steps=10,
    units=128,
    learning_rate=0.0005,
    batch_size=32,
    max_epochs=100
):
    """Train one LSTM configuration and return validation results."""

    # Create training sequences.
    x_train, y_train = create_sequences(
        train_features,
        train_targets,
        time_steps=time_steps
    )

    # Add the last training observations to the validation context.
    validation_features_with_context = np.concatenate(
        [
            train_features[-time_steps:],
            validation_features
        ],
        axis=0
    )

    validation_targets_with_context = np.concatenate(
        [
            train_targets[-time_steps:],
            validation_targets
        ],
        axis=0
    )

    x_validation, y_validation = create_sequences(
        validation_features_with_context,
        validation_targets_with_context,
        time_steps=time_steps
    )

    # Clear the previous model before starting another experiment.
    K.clear_session()
    set_random_seed(42)

    model = build_lstm_model(
        input_shape=(
            x_train.shape[1],
            x_train.shape[2]
        ),
        units=units,
        output_size=y_train.shape[1],
        learning_rate=learning_rate
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        x_train,
        y_train,
        validation_data=(
            x_validation,
            y_validation
        ),
        epochs=max_epochs,
        batch_size=batch_size,
        shuffle=False,
        callbacks=[early_stopping],
        verbose=0
    )

    return {
        "time_steps": time_steps,
        "units": units,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs_completed": len(history.history["loss"]),
        "best_training_loss": min(history.history["loss"]),
        "best_validation_loss": min(history.history["val_loss"])
    }

In [ ]:
time_step_results = []

for time_steps in [10, 30, 60]:
    result = run_experiment(
        train_features=train_features,
        train_targets=train_targets,
        validation_features=validation_features,
        validation_targets=validation_targets,
        time_steps=time_steps,
        units=128,
        learning_rate=0.0005,
        batch_size=32
    )

    time_step_results.append(result)

time_step_results_df = pd.DataFrame(time_step_results)

time_step_results_df.sort_values(
    "best_validation_loss"
)

In [ ]:
unit_results = []

for units in [32, 64, 128, 256]:
    result = run_experiment(
        train_features=train_features,
        train_targets=train_targets,
        validation_features=validation_features,
        validation_targets=validation_targets,
        time_steps=10,
        units=units,
        learning_rate=0.0005,
        batch_size=32
    )

    unit_results.append(result)

unit_results_df = pd.DataFrame(unit_results)

unit_results_df.sort_values(
    "best_validation_loss"
)

In [ ]:
learning_rate_results = []

for learning_rate in [0.001, 0.0005, 0.0001]:
    result = run_experiment(
        train_features=train_features,
        train_targets=train_targets,
        validation_features=validation_features,
        validation_targets=validation_targets,
        time_steps=10,
        units=128,
        learning_rate=learning_rate,
        batch_size=32
    )

    learning_rate_results.append(result)

learning_rate_results_df = pd.DataFrame(
    learning_rate_results
)

learning_rate_results_df.sort_values(
    "best_validation_loss"
)

In [ ]:
batch_size_results = []

for batch_size in [16, 32, 64, 128]:
    result = run_experiment(
        train_features=train_features,
        train_targets=train_targets,
        validation_features=validation_features,
        validation_targets=validation_targets,
        time_steps=10,
        units=128,
        learning_rate=0.0005,
        batch_size=batch_size
    )

    batch_size_results.append(result)

batch_size_results_df = pd.DataFrame(
    batch_size_results
)

batch_size_results_df.sort_values(
    "best_validation_loss"
)

In [ ]:
all_results = pd.concat(
    [
        time_step_results_df,
        unit_results_df,
        learning_rate_results_df,
        batch_size_results_df
    ],
    ignore_index=True
)

all_results.to_csv(
    PROJECT_ROOT /
    "results/hyperparameter_results.csv",
    index=False
)

all_results

In [ ]:
import matplotlib.pyplot as plt


def plot_parameter_results(
    results_df,
    parameter,
    title
):
    summary = (
        results_df
        .groupby(parameter, as_index=False)
        ["best_validation_loss"]
        .min()
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        summary[parameter],
        summary["best_validation_loss"],
        marker="o"
    )

    plt.title(title)
    plt.xlabel(parameter)
    plt.ylabel("Best Validation Loss")
    plt.grid(alpha=0.3)
    plt.show()